# Data Preprocessing ABSA Hotel Santika

Notebook ini menyatukan seluruh tahap awal data pipeline dalam satu file:

1. Merge dataset dari folder scraping Agoda, Tiket, dan Traveloka.
2. EDA unprocessed pada dataset raw hasil merge.
3. Data preprocessing sampai menjadi dataset clean.
4. EDA processed dan WordCloud pada dataset clean.

Guardrail jumlah baris:
- Raw dataset: 17.868 review.
- Clean dataset: 14.747 review.


### Install Library Pendukung


In [ ]:
!pip install -q lingua-language-detector deep-translator wordcloud openpyxl


### Import Library dan Setup Dasar


In [ ]:
import pandas as pd
import os
import re
import time
from datetime import datetime, timedelta
from lingua import Language, LanguageDetectorBuilder
from deep_translator import GoogleTranslator

# Whitelist bahasa yang realistis untuk hotel di Indonesia.
# Membatasi kandidat = akurasi naik signifikan untuk teks pendek.
ALLOWED_LANGS = [
    Language.INDONESIAN,
    Language.MALAY,        # sering tertukar dengan Indonesia
    Language.ENGLISH,
    Language.JAPANESE,
    Language.KOREAN,
    Language.CHINESE,
    Language.ARABIC,
    Language.TAGALOG,
    Language.THAI,
    Language.VIETNAMESE,
    Language.DUTCH,
    Language.FRENCH,
    Language.GERMAN,
    Language.SPANISH,
    Language.ITALIAN,
    Language.PORTUGUESE,
    Language.RUSSIAN,
    Language.HINDI,
    Language.TURKISH,
]

LANG_DETECTOR = (
    LanguageDetectorBuilder
    .from_languages(*ALLOWED_LANGS)
    .with_preloaded_language_models()
    .with_minimum_relative_distance(0.0)
    .build()
)

# Kode ISO 639-1 untuk mapping ke string singkat
LANG_TO_CODE = {
    Language.INDONESIAN: 'id',
    Language.MALAY: 'id',          # Malay diperlakukan sebagai Indonesia
    Language.ENGLISH: 'en',
    Language.JAPANESE: 'ja',
    Language.KOREAN: 'ko',
    Language.CHINESE: 'zh',
    Language.ARABIC: 'ar',
    Language.TAGALOG: 'tl',
    Language.THAI: 'th',
    Language.VIETNAMESE: 'vi',
    Language.DUTCH: 'nl',
    Language.FRENCH: 'fr',
    Language.GERMAN: 'de',
    Language.SPANISH: 'es',
    Language.ITALIAN: 'it',
    Language.PORTUGUESE: 'pt',
    Language.RUSSIAN: 'ru',
    Language.HINDI: 'hi',
    Language.TURKISH: 'tr',
}

print('All imports loaded ✓')


### Konfigurasi Path dan Sumber Data


In [ ]:
from pathlib import Path
import pandas as pd
PROJECT_LOCAL_ROOT = Path(r'C:\Users\cencen04_\Downloads\ABSA Hotel Santika')
KAGGLE_INPUT = Path('/kaggle/input')
KAGGLE_WORKING = Path('/kaggle/working')
IS_KAGGLE = KAGGLE_INPUT.exists() and KAGGLE_WORKING.exists()
OUTPUT_DIR = KAGGLE_WORKING if IS_KAGGLE else PROJECT_LOCAL_ROOT / 'Data Preprocessing'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def resolve_csv(filename, keywords=()):
    direct_candidates = [
        OUTPUT_DIR / filename,
        PROJECT_LOCAL_ROOT / 'Data Preprocessing' / filename,
        PROJECT_LOCAL_ROOT / 'Data Preprocessing' / 'Merge dan Wordcloud' / filename,
    ]
    for path in direct_candidates:
        if path.exists():
            print('Loaded:', path)
            return path

    candidates = []
    search_roots = [KAGGLE_INPUT, KAGGLE_WORKING, PROJECT_LOCAL_ROOT / 'Data Preprocessing', Path.cwd()]
    for root in search_roots:
        root = Path(root)
        if root.exists():
            candidates.extend(root.rglob(filename))

    def score(p):
        low = str(p).lower()
        value = sum(str(k).lower() in low for k in keywords)
        if filename.lower() in low:
            value += 2
        return value

    candidates = sorted(set(candidates), key=lambda p: (-score(p), str(p).lower()))
    if not candidates:
        raise FileNotFoundError(f'Tidak menemukan {filename}. Jalankan data_preprocessing.ipynb dulu atau upload outputnya ke Kaggle.')
    print('Loaded:', candidates[0])
    return candidates[0]

BASE_DIR = str(OUTPUT_DIR)
OUTPUT_FILE_RAW = 'dataset_absa_santika_raw.csv'
OUTPUT_FILE_CLEAN = 'dataset_absa_santika_clean.csv'
EXPECTED_RAW_ROWS = 17868
EXPECTED_CLEAN_ROWS = 14747
TRAVELOKA_SCRAPE_DATE = datetime(2026, 5, 27)

def has_platform_dirs(path):
    path = Path(path)
    return all((path / p).exists() for p in ['Agoda', 'Tiket', 'Traveloka'])

def resolve_scraping_dir():
    candidates = [PROJECT_LOCAL_ROOT / 'Data Scraping', Path.cwd(), Path.cwd() / 'Data Scraping', Path.cwd() / 'data_scraping']
    if KAGGLE_INPUT.exists():
        for root in KAGGLE_INPUT.iterdir():
            candidates += [root, root / 'Data Scraping', root / 'data_scraping', root / 'data-scraping']
    for c in candidates:
        if has_platform_dirs(c): return Path(c)
    raise FileNotFoundError('Upload folder Kaggle yang berisi Agoda, Tiket, dan Traveloka.')

SCRAPING_DIR = resolve_scraping_dir()
print('Scraping source:', SCRAPING_DIR)
for p in ['Agoda','Tiket','Traveloka']:
    print(p, len(list((SCRAPING_DIR / p).glob('*.csv'))), 'CSV')


### Definisi Helper Merge dan Parsing Tanggal


In [ ]:
def normalize_hotel_name(filename):
    """Ekstrak & normalisasi nama hotel dari nama file CSV."""
    name = os.path.splitext(filename)[0]
    for prefix in ['Hotel ', 'hotel_', 'Hotel_']:
        if name.startswith(prefix):
            name = name[len(prefix):]
    name = name.strip()
    name_lower = name.lower()

    if 'bandung' in name_lower:
        return 'Hotel Santika Bandung'
    elif 'bekasi' in name_lower or 'megacity' in name_lower or 'mega city' in name_lower:
        return 'Hotel Santika Mega City Bekasi'
    elif 'bogor' in name_lower:
        return 'Hotel Santika Bogor'
    elif 'cirebon' in name_lower:
        return 'Hotel Santika Cirebon'
    elif 'depok' in name_lower:
        return 'Hotel Santika Depok'
    elif 'tasik' in name_lower:
        return 'Hotel Santika Tasikmalaya'
    else:
        return f'Hotel Santika {name}'


def convert_traveloka_date(relative_str, scrape_date):
    """Konversi 'Diulas X minggu/hari lalu' ke datetime."""
    if pd.isna(relative_str) or not isinstance(relative_str, str):
        return pd.NaT
    text = relative_str.strip().lower()

    match = re.match(r'diulas\s+(\d+)\s+minggu\s+lalu', text)
    if match:
        return scrape_date - timedelta(weeks=int(match.group(1)))

    match = re.match(r'diulas\s+(\d+)\s+hari\s+lalu', text)
    if match:
        return scrape_date - timedelta(days=int(match.group(1)))

    match = re.match(r'diulas\s+(\d+)\s+bulan\s+lalu', text)
    if match:
        return scrape_date - timedelta(days=int(match.group(1)) * 30)

    return pd.NaT


def parse_agoda_date(date_str):
    """Parse format Agoda: 'May 17, 2026'"""
    if pd.isna(date_str) or not isinstance(date_str, str):
        return pd.NaT
    try:
        return pd.to_datetime(date_str.strip(), format='%B %d, %Y')
    except:
        try:
            return pd.to_datetime(date_str.strip())
        except:
            return pd.NaT


def parse_tiket_date(date_str):
    """Parse format Tiket: '19 May 2026'"""
    if pd.isna(date_str) or not isinstance(date_str, str):
        return pd.NaT
    try:
        return pd.to_datetime(date_str.strip(), format='%d %b %Y')
    except:
        try:
            return pd.to_datetime(date_str.strip())
        except:
            return pd.NaT


def find_column(df, target_name):
    """Cari kolom (handle dash vs underscore)."""
    for col in df.columns:
        if col.lower().replace('-', '_') == target_name.lower():
            return col
    return None


print('Helper functions loaded ✓')


### Membaca CSV Per Platform


In [ ]:
def process_platform(base_dir, platform, date_parser, scrape_date=None):
    """Baca semua CSV dari folder platform, return DataFrame."""
    folder = os.path.join(base_dir, platform)
    if not os.path.exists(folder):
        print(f'  ❌ Folder {platform} tidak ditemukan: {folder}')
        return pd.DataFrame()

    all_rows = []
    for fname in sorted(os.listdir(folder)):
        if not fname.endswith('.csv'):
            continue
        fpath = os.path.join(folder, fname)
        hotel_name = normalize_hotel_name(fname)
        df = pd.read_csv(fpath, encoding='utf-8-sig')

        text_col = find_column(df, 'teks_ulasan')
        date_col = find_column(df, 'tanggal')

        if text_col is None:
            print(f'  ⚠️  [SKIP] Kolom teks ulasan tidak ditemukan di {fname}: {df.columns.tolist()}')
            continue

        for _, row in df.iterrows():
            text = str(row.get(text_col, '')).strip() if pd.notna(row.get(text_col)) else ''
            date_raw = row.get(date_col, '') if date_col else ''

            if scrape_date:
                date_parsed = date_parser(date_raw, scrape_date)
            else:
                date_parsed = date_parser(date_raw)

            all_rows.append({
                'platform': platform,
                'hotel_name': hotel_name,
                'text_review': text,
                'date': date_parsed,
            })

        print(f'  ✅ {platform}/{fname}: {len(df)} baris → {hotel_name}')

    return pd.DataFrame(all_rows)


# --- Proses Agoda ---
print('📂 Membaca data Agoda...')
df_agoda = process_platform(SCRAPING_DIR, 'Agoda', parse_agoda_date)
print(f'   Total Agoda: {len(df_agoda)} baris\n')

# --- Proses Tiket ---
print('📂 Membaca data Tiket...')
df_tiket = process_platform(SCRAPING_DIR, 'Tiket', parse_tiket_date)
print(f'   Total Tiket: {len(df_tiket)} baris\n')

# --- Proses Traveloka ---
print('📂 Membaca data Traveloka (+ konversi tanggal relatif)...')
df_traveloka = process_platform(SCRAPING_DIR, 'Traveloka', convert_traveloka_date, TRAVELOKA_SCRAPE_DATE)
print(f'   Total Traveloka: {len(df_traveloka)} baris')


### Menggabungkan Raw Dataset


In [ ]:
# Gabungkan semua platform
df_all = pd.concat([df_agoda, df_tiket, df_traveloka], ignore_index=True)
print(f'Total baris sebelum cleaning: {len(df_all):,}')

# Hapus review dengan teks kosong
df_all = df_all[df_all['text_review'].str.strip().astype(bool)]
print(f'Setelah hapus review kosong:  {len(df_all):,}')

# Hapus baris tanpa tanggal
before = len(df_all)
df_all = df_all.dropna(subset=['date'])
print(f'Setelah hapus tanggal kosong: {len(df_all):,} (hapus {before - len(df_all):,})')

# Reset index
df_all = df_all.reset_index(drop=True)

print(f'\nDataset siap untuk deteksi bahasa: {len(df_all):,} review')
if len(df_all) != EXPECTED_RAW_ROWS:
    raise ValueError(f'Raw row count mismatch before language processing: expected {EXPECTED_RAW_ROWS:,}, actual {len(df_all):,}')


### Deteksi Bahasa Awal


In [ ]:
def detect_language(text):
    """Deteksi bahasa pakai lingua. Return kode ISO 639-1 atau 'unknown'."""
    if not isinstance(text, str):
        return 'unknown'
    text_clean = text.strip()
    if len(text_clean) < 10:
        return 'unknown'
    try:
        # Cek confidence — kalau ragu, return 'unknown'
        confidence = LANG_DETECTOR.compute_language_confidence_values(text_clean)
        if not confidence:
            return 'unknown'
        top = confidence[0]
        # Skor terlalu rendah → tidak yakin
        if top.value < 0.50:
            return 'unknown'
        return LANG_TO_CODE.get(top.language, 'unknown')
    except Exception:
        return 'unknown'


print('Mendeteksi bahasa setiap review pakai lingua...')
print('(Estimasi: ~1-2 menit untuk ~17.000 review)\n')

start_time = time.time()
df_all['original_language'] = df_all['text_review'].apply(detect_language)
elapsed = time.time() - start_time

print(f'Deteksi bahasa selesai dalam {elapsed:.1f} detik\n')

# Distribusi bahasa
lang_dist = df_all['original_language'].value_counts()
print('Distribusi bahasa:')
LABEL_MAP = {
    'id': 'Indonesia', 'en': 'English', 'ja': 'Japanese', 'ko': 'Korean',
    'zh': 'Chinese', 'ar': 'Arabic', 'tl': 'Tagalog', 'th': 'Thai',
    'vi': 'Vietnamese', 'nl': 'Dutch', 'fr': 'French', 'de': 'German',
    'es': 'Spanish', 'it': 'Italian', 'pt': 'Portuguese', 'ru': 'Russian',
    'hi': 'Hindi', 'tr': 'Turkish', 'unknown': 'Tidak terdeteksi/terlalu pendek',
}
for lang, count in lang_dist.items():
    pct = count / len(df_all) * 100
    label = LABEL_MAP.get(lang, lang)
    print(f'  {lang:>7s} ({label}): {count:,} review ({pct:.1f}%)')


print('Language detection summary:')
print(df_all['original_language'].value_counts().to_string())


### Menentukan Review yang Perlu Diterjemahkan


In [ ]:
# Backup teks asli SEBELUM translate (untuk transparansi)
df_all['text_review_original'] = df_all['text_review'].copy()

# Identifikasi review yang perlu ditranslate: semua selain id & unknown
LANGS_TO_TRANSLATE = [l for l in df_all['original_language'].unique()
                      if l not in ('id', 'unknown')]
needs_translation = df_all['original_language'].isin(LANGS_TO_TRANSLATE)
total_to_translate = needs_translation.sum()

print(f'Review yang perlu ditranslate (non-id → id): {total_to_translate:,}')
print(f'Bahasa yang akan ditranslate: {sorted(LANGS_TO_TRANSLATE)}')
print(f'Review bahasa Indonesia (skip): {(df_all["original_language"] == "id").sum():,}')
print(f'Review unknown (skip): {(df_all["original_language"] == "unknown").sum():,}')

# Breakdown jumlah per bahasa yang akan ditranslate
print('\nDistribusi review yang akan ditranslate:')
print(df_all[needs_translation]['original_language'].value_counts().to_string())


### Menerjemahkan Review Non-Indonesia


In [ ]:
# Path checkpoint di Google Drive
CHECKPOINT_PATH = os.path.join(BASE_DIR, '_translation_checkpoint.csv')


def translate_one(text, src_lang, target='id'):
    """Translate single text. Auto-detect source jika src_lang gagal."""
    if not text or len(str(text).strip()) == 0:
        return text
    text = str(text)[:4900]  # deep-translator max 5000 chars

    try:
        # Coba dengan source lang yang terdeteksi
        translator = GoogleTranslator(source=src_lang, target=target)
        result = translator.translate(text)
        return result if result else text
    except Exception:
        # Fallback ke auto-detect
        try:
            translator = GoogleTranslator(source='auto', target=target)
            result = translator.translate(text)
            return result if result else text
        except Exception:
            return text  # Gagal total → kembalikan teks asli


def translate_indices(df, indices, batch_delay=0.3):
    """Translate review pada indices tertentu dari df. Return dict {idx: translated}."""
    results = {}
    total = len(indices)

    for i, idx in enumerate(indices, 1):
        text = df.at[idx, 'text_review']
        src_lang = df.at[idx, 'original_language']

        # Mapping ke kode bahasa Google Translate
        # Sebagian besar kode ISO 639-1 sudah kompatibel dengan Google Translate
        gt_src = src_lang if src_lang not in ('unknown',) else 'auto'

        translated = translate_one(text, gt_src, target='id')
        results[idx] = translated

        # Progress
        if i % 10 == 0 or i == total:
            pct = i / total * 100
            print(f'  Translated: {i:,}/{total:,} ({pct:.1f}%)', end='\r')

        # Rate limiting
        time.sleep(batch_delay)

    print()  # New line setelah progress
    return results


# === MULAI TRANSLATE ===
if total_to_translate > 0:
    en_indices = df_all[needs_translation].index.tolist()

    # Cek checkpoint
    if os.path.exists(CHECKPOINT_PATH):
        print(f'📂 Checkpoint ditemukan, melanjutkan dari checkpoint...')
        df_checkpoint = pd.read_csv(CHECKPOINT_PATH, encoding='utf-8-sig')
        translated_map = dict(zip(df_checkpoint['index'].astype(int),
                                  df_checkpoint['translated']))
        remaining_indices = [idx for idx in en_indices if idx not in translated_map]
        print(f'   Sudah ditranslate: {len(translated_map):,}')
        print(f'   Tersisa: {len(remaining_indices):,}')
    else:
        translated_map = {}
        remaining_indices = en_indices

    if len(remaining_indices) > 0:
        print(f'\n🔄 Menerjemahkan {len(remaining_indices):,} review (non-id → id)...')
        print(f'   Estimasi waktu: ~{len(remaining_indices) * 0.5 / 60:.0f} menit')
        print(f'   Checkpoint disimpan setiap 200 review\n')

        # Translate dalam chunk dengan auto-checkpoint setiap 200 review
        CHECKPOINT_EVERY = 200
        for chunk_start in range(0, len(remaining_indices), CHECKPOINT_EVERY):
            chunk = remaining_indices[chunk_start:chunk_start + CHECKPOINT_EVERY]
            chunk_results = translate_indices(df_all, chunk, batch_delay=0.3)
            translated_map.update(chunk_results)

            # Auto-save checkpoint
            df_ckpt = pd.DataFrame([
                {'index': k, 'translated': v} for k, v in translated_map.items()
            ])
            df_ckpt.to_csv(CHECKPOINT_PATH, index=False, encoding='utf-8-sig')
            print(f'  💾 Checkpoint disimpan ({len(translated_map):,} total)')

    # Apply translations ke kolom text_review (kolom text_review_original tetap asli)
    for idx, translated in translated_map.items():
        if idx in df_all.index:
            df_all.at[idx, 'text_review'] = translated

    print(f'\n✅ Translate selesai! {len(translated_map):,} review berhasil diterjemahkan.')
else:
    print('Tidak ada review non-Indonesia, skip translate.')


### Menyusun Format Raw Dataset


In [ ]:
# Tambahkan review_id
df_all = df_all.reset_index(drop=True)
df_all.insert(0, 'review_id', range(1, len(df_all) + 1))

# Format tanggal → YYYY-MM-DD
df_all['date'] = pd.to_datetime(df_all['date']).dt.strftime('%Y-%m-%d')

# Urutkan kolom
df_all = df_all[[
    'review_id', 'platform', 'hotel_name',
    'text_review', 'text_review_original',
    'date', 'original_language',
]]

print(f'Dataset final: {len(df_all):,} review')
print(f'Kolom: {df_all.columns.tolist()}')
df_all.head(10)


### Deteksi Bahasa Final dan Koreksi Label Bahasa


In [ ]:
# ====== PATCH CELL 2: Re-detect bahasa dengan threshold lebih longgar + heuristic ======

# Setup detector ulang (standalone, kalau runtime sudah restart)
ALLOWED_LANGS = [
    Language.INDONESIAN, Language.MALAY, Language.ENGLISH,
    Language.JAPANESE, Language.KOREAN, Language.CHINESE,
    Language.ARABIC, Language.TAGALOG, Language.THAI,
    Language.VIETNAMESE, Language.DUTCH, Language.FRENCH,
    Language.GERMAN, Language.SPANISH, Language.ITALIAN,
    Language.PORTUGUESE, Language.RUSSIAN, Language.HINDI,
    Language.TURKISH,
]
LANG_DETECTOR = LanguageDetectorBuilder.from_languages(*ALLOWED_LANGS).with_preloaded_language_models().build()
LANG_TO_CODE = {
    Language.INDONESIAN: 'id', Language.MALAY: 'id', Language.ENGLISH: 'en',
    Language.JAPANESE: 'ja', Language.KOREAN: 'ko', Language.CHINESE: 'zh',
    Language.ARABIC: 'ar', Language.TAGALOG: 'tl', Language.THAI: 'th',
    Language.VIETNAMESE: 'vi', Language.DUTCH: 'nl', Language.FRENCH: 'fr',
    Language.GERMAN: 'de', Language.SPANISH: 'es', Language.ITALIAN: 'it',
    Language.PORTUGUESE: 'pt', Language.RUSSIAN: 'ru', Language.HINDI: 'hi',
    Language.TURKISH: 'tr',
}

# Kata kunci khas bahasa Indonesia. Jika teks mengandung >=2 kata ini, treat as 'id'.
ID_KEYWORDS = {
    'yang', 'saya', 'tidak', 'sangat', 'dengan', 'untuk', 'sudah', 'juga',
    'bagus', 'enak', 'ramah', 'bersih', 'nyaman', 'kamar', 'hotel', 'staff',
    'staf', 'pelayanan', 'sarapan', 'kolam', 'pemandangan', 'lokasi', 'mantap',
    'menyenangkan', 'membantu', 'gak', 'aja', 'kayak', 'banget', 'lagi', 'kalo',
    'kalau', 'biasa', 'cukup', 'kurang', 'lebih', 'pernah', 'banyak', 'sekali',
    'kembali', 'baik', 'baru', 'lama', 'tempat', 'depan', 'dekat', 'jauh',
    'breakfast',  # umum dipakai di review hotel Indonesia
}

def is_indonesian_by_keywords(text, min_hits=2):
    """Cek apakah teks 'cukup Indonesia' berdasarkan kata kunci."""
    if not isinstance(text, str):
        return False
    words = re.findall(r'\b[a-zA-Z]+\b', text.lower())
    hits = sum(1 for w in words if w in ID_KEYWORDS)
    return hits >= min_hits


def detect_language_v2(text):
    """Re-detect bahasa dengan threshold 0.30 + heuristic kata kunci Indonesia."""
    if not isinstance(text, str):
        return 'unknown'
    text_clean = text.strip()
    if len(text_clean) < 10:
        # Untuk teks sangat pendek, cek apakah kata Indonesia
        if is_indonesian_by_keywords(text_clean, min_hits=1):
            return 'id'
        return 'unknown'

    try:
        confidence = LANG_DETECTOR.compute_language_confidence_values(text_clean)
        if not confidence:
            # Fallback ke heuristic
            return 'id' if is_indonesian_by_keywords(text_clean) else 'unknown'

        top = confidence[0]
        top_code = LANG_TO_CODE.get(top.language, 'unknown')

        # Confidence cukup tinggi → trust hasilnya
        if top.value >= 0.30:
            # SPECIAL CASE: kalau hasil 'tl' (Tagalog), cross-check dengan kata Indonesia
            # Karena Tagalog & Indonesia sering campur Inggris, mudah tertukar
            if top_code == 'tl' and is_indonesian_by_keywords(text_clean):
                return 'id'
            return top_code

        # Confidence rendah → fallback ke heuristic
        if is_indonesian_by_keywords(text_clean):
            return 'id'

        # Kalau hasil top adalah 'id' meskipun confidence rendah, masih reasonable
        if top_code == 'id' and top.value >= 0.15:
            return 'id'

        return 'unknown'
    except Exception:
        return 'id' if is_indonesian_by_keywords(text_clean) else 'unknown'


# Re-detect berdasarkan text_review_original (teks asli sebelum translate)
print('Re-detect bahasa (threshold 0.30 + heuristic kata kunci ID)...')
print(f'Source kolom: text_review_original (teks asli, bukan hasil translate)\n')

start = time.time()
df_all['original_language_new'] = df_all['text_review_original'].apply(detect_language_v2)
elapsed = time.time() - start

print(f'Selesai dalam {elapsed:.1f} detik\n')

# Bandingkan
print('Distribusi bahasa SETELAH patch:')
print(df_all['original_language_new'].value_counts().to_string())
print()

# Berapa label yang berubah
changed = (df_all['original_language'] != df_all['original_language_new']).sum()
print(f'Total label yang BERUBAH: {changed:,} review')
print()
print('Detail perubahan label:')
change_table = pd.crosstab(df_all['original_language'], df_all['original_language_new'], margins=True)
print(change_table.to_string())


### Rollback Teks untuk Review Indonesia


In [ ]:
# ====== PATCH CELL 3: Rollback text_review untuk row yang sekarang id/unknown ======
# Kalau label baru = 'id' atau 'unknown' tapi text_review sudah ditranslate (beda dari original),
# itu artinya translate sebelumnya salah → balikkan ke text_review_original

# Apply label baru
df_all['original_language'] = df_all['original_language_new']
df_all = df_all.drop(columns=['original_language_new'])

# Rollback
ROLLBACK_LANGS = ['id', 'unknown']
needs_rollback = (
    df_all['original_language'].isin(ROLLBACK_LANGS)
    & (df_all['text_review'] != df_all['text_review_original'])
)
n_rollback = needs_rollback.sum()
print(f'Review yang di-rollback (text_review → text_review_original): {n_rollback:,}')

if n_rollback > 0:
    print('\nSample rollback (sebelum rollback):')
    sample = df_all[needs_rollback].head(5)
    for _, r in sample.iterrows():
        print(f'  [{r["original_language"]}] (sebelum) translate: "{r["text_review"][:80]}"')
        print(f'         (sesudah) original : "{r["text_review_original"][:80]}"')
        print()

    df_all.loc[needs_rollback, 'text_review'] = df_all.loc[needs_rollback, 'text_review_original']
    print(f'✅ {n_rollback:,} review berhasil di-rollback ke teks asli')
else:
    print('Tidak ada yang perlu di-rollback.')


### Translate Tambahan Setelah Koreksi Bahasa


In [ ]:
# ====== PATCH CELL 4: Translate review yang BARU terdeteksi non-id (kalau ada) ======
# Skenario: review yang sebelumnya 'unknown' ternyata bahasa asing dan belum pernah ditranslate

from deep_translator import GoogleTranslator

LANGS_TO_TRANSLATE = [l for l in df_all['original_language'].unique()
                      if l not in ('id', 'unknown')]

# Cari row yang labelnya non-id non-unknown TAPI text_review masih sama dengan original (belum ditranslate)
needs_new_translate = (
    df_all['original_language'].isin(LANGS_TO_TRANSLATE)
    & (df_all['text_review'] == df_all['text_review_original'])
)
n_new = needs_new_translate.sum()
print(f'Review baru yang perlu ditranslate: {n_new:,}')
print(f'(Bahasa: {df_all[needs_new_translate]["original_language"].value_counts().to_dict()})')

if n_new > 0:
    print(f'\n🔄 Menerjemahkan {n_new:,} review baru...')

    def translate_one(text, src, target='id'):
        if not text or len(str(text).strip()) == 0:
            return text
        text = str(text)[:4900]
        try:
            return GoogleTranslator(source=src, target=target).translate(text) or text
        except Exception:
            try:
                return GoogleTranslator(source='auto', target=target).translate(text) or text
            except Exception:
                return text

    indices_to_translate = df_all[needs_new_translate].index.tolist()
    for i, idx in enumerate(indices_to_translate, 1):
        text = df_all.at[idx, 'text_review_original']
        src = df_all.at[idx, 'original_language']
        translated = translate_one(text, src, target='id')
        df_all.at[idx, 'text_review'] = translated

        if i % 10 == 0 or i == n_new:
            print(f'  Progress: {i}/{n_new} ({i/n_new*100:.0f}%)', end='\r')
        time.sleep(0.3)
    print(f'\n✅ {n_new:,} review baru berhasil diterjemahkan')
else:
    print('Tidak ada review baru yang perlu ditranslate.')


### Menyimpan Raw Dataset


In [ ]:
# Simpan raw dataset hasil merge + language processing
raw_output_path = os.path.join(BASE_DIR, OUTPUT_FILE_RAW)
df_all.to_csv(raw_output_path, index=False, encoding='utf-8-sig')
print(f'Raw dataset saved: {raw_output_path}')
print(f'Raw rows: {len(df_all):,}')
if len(df_all) != EXPECTED_RAW_ROWS:
    raise ValueError(f'Raw row count mismatch: expected {EXPECTED_RAW_ROWS:,}, actual {len(df_all):,}')
if os.path.exists(CHECKPOINT_PATH):
    os.remove(CHECKPOINT_PATH)
print('Checkpoint: raw dataset ready. Run eda_unprocessed.ipynb here if needed, then continue preprocessing cells below.')


## EDA Unprocessed Dataset

Tahap ini dilakukan setelah raw dataset terbentuk. Dataset belum melewati cleaning teks, sehingga EDA ini dipakai untuk memahami kondisi data awal hasil scraping dan merge.


### Load Raw Dataset untuk EDA Unprocessed


In [ ]:
import matplotlib.pyplot as plt

raw_df = df_all.copy()
EXPECTED_RAW_ROWS = 17868
print('=== EDA Unprocessed Dataset ===')
print(f'Total review raw: {len(raw_df):,}')
print(f'Kolom: {raw_df.columns.tolist()}')
if len(raw_df) != EXPECTED_RAW_ROWS:
    print(f'[WARN] Expected {EXPECTED_RAW_ROWS:,}, actual {len(raw_df):,}')

display(raw_df.head())


### Overview Raw Dataset


In [ ]:
print('=== Overview Raw Dataset ===')
print(f'Platform: {raw_df["platform"].nunique()}')
print(f'Hotel: {raw_df["hotel_name"].nunique()}')

raw_dates = pd.to_datetime(raw_df['date'], errors='coerce')
print(f'Rentang tanggal: {raw_dates.min().date()} s/d {raw_dates.max().date()}')
print(f'Tanggal invalid: {raw_dates.isna().sum():,}')

print('\nMissing values:')
display(raw_df.isna().sum().to_frame('missing_count'))


### Distribusi Platform dan Hotel Raw


In [ ]:
raw_platform_summary = raw_df.groupby('platform').size().to_frame('jumlah_review').reset_index().sort_values('jumlah_review', ascending=False)
raw_hotel_summary = raw_df.groupby('hotel_name').size().to_frame('jumlah_review').reset_index().sort_values('jumlah_review', ascending=False)
raw_platform_hotel = pd.crosstab(raw_df['hotel_name'], raw_df['platform'], margins=True)

print('--- Distribusi Platform - Raw ---')
display(raw_platform_summary)

print('--- Distribusi Hotel - Raw ---')
display(raw_hotel_summary)

print('--- Platform x Hotel - Raw ---')
display(raw_platform_hotel)

if 'original_language' in raw_df.columns:
    raw_language_summary = raw_df.groupby('original_language').size().to_frame('jumlah_review').reset_index().sort_values('jumlah_review', ascending=False)
    print('--- Bahasa Asli - Raw ---')
    display(raw_language_summary)


### Kualitas Teks Raw


In [ ]:
raw_text_len = raw_df['text_review'].fillna('').astype(str).str.len()
raw_quality_summary = pd.DataFrame({
    'metric': [
        'min_len', 'mean_len', 'median_len', 'max_len',
        'duplicate_text', 'short_review_lt_20', 'empty_text'
    ],
    'value': [
        raw_text_len.min(),
        round(raw_text_len.mean(), 2),
        raw_text_len.median(),
        raw_text_len.max(),
        raw_df['text_review'].duplicated().sum(),
        (raw_text_len < 20).sum(),
        raw_df['text_review'].fillna('').astype(str).str.strip().eq('').sum(),
    ]
})
display(raw_quality_summary)


### Visualisasi Tahun dan Platform Raw


In [ ]:
raw_years = pd.to_datetime(raw_df['date'], errors='coerce').dt.year.value_counts().sort_index()
ax = raw_years.plot(kind='bar', figsize=(12, 5), color='#4472C4')
ax.set_title('Distribusi Review per Tahun - Unprocessed')
ax.set_xlabel('Tahun')
ax.set_ylabel('Jumlah Review')
plt.tight_layout()
plt.show()

ax = raw_platform_summary.set_index('platform')['jumlah_review'].plot(kind='bar', figsize=(8, 4), color='#70AD47')
ax.set_title('Distribusi Review per Platform - Unprocessed')
ax.set_xlabel('Platform')
ax.set_ylabel('Jumlah Review')
plt.tight_layout()
plt.show()


### Export Ringkasan EDA Unprocessed


In [ ]:
raw_summary_path = Path(BASE_DIR) / 'eda_unprocessed_summary.xlsx'
with pd.ExcelWriter(raw_summary_path) as writer:
    raw_platform_summary.to_excel(writer, index=False, sheet_name='platform')
    raw_hotel_summary.to_excel(writer, index=False, sheet_name='hotel')
    raw_platform_hotel.to_excel(writer, sheet_name='platform_x_hotel')
    raw_quality_summary.to_excel(writer, index=False, sheet_name='quality')
    if 'raw_language_summary' in globals():
        raw_language_summary.to_excel(writer, index=False, sheet_name='language')
print('Saved:', raw_summary_path)


## Preprocessing Clean Dataset


### Menyiapkan Input Preprocessing


In [ ]:
# Mulai preprocessing dari raw dataset hasil merge
MIN_LENGTH = 20
initial_count = len(df_all)
df = df_all.copy()
emoji_pattern = re.compile('[\U0001F600-\U0001F64F\U0001F300-\U0001F5FF\U0001F680-\U0001F6FF\U0001F1E0-\U0001F1FF\U00002702-\U000027B0\U0001f900-\U0001f9FF\U00002600-\U000026FF\u200d\ufe0f]')
print(f'Preprocessing input rows: {initial_count:,}')


### Menghapus Duplikat Exact


In [ ]:
before = len(df)
df = df.drop_duplicates(subset='text_review', keep='first')
removed = before - len(df)
print(f'Hapus duplikat: {removed:,} review dihapus')
print(f'Sisa: {len(df):,} review')


### Filter Review Terlalu Pendek


In [ ]:
MIN_LENGTH = 20

# Tampilkan sample yang akan dihapus
short = df[df['text_review'].astype(str).str.len() < MIN_LENGTH]
print(f'Review < {MIN_LENGTH} char: {len(short):,}')
print('\nSample yang dihapus:')
for _, row in short.head(10).iterrows():
    print(f'  [{row["platform"]}] "{row["text_review"]}"')

before = len(df)
df = df[df['text_review'].astype(str).str.len() >= MIN_LENGTH]
removed = before - len(df)
print(f'\nDihapus: {removed:,} review')
print(f'Sisa: {len(df):,} review')


### Normalisasi Teks Dasar


In [ ]:
# Mapping emoji ke teks sentimen (opsional, bisa dibuang juga)
EMOJI_SENTIMENT = {
    '😀': '', '😃': '', '😄': '', '😁': '', '😆': '',
    '😅': '', '🤣': '', '😂': '', '🙂': '', '😊': '',
    '😍': '', '🥰': '', '😘': '', '😗': '', '😙': '',
    '😚': '', '😋': '', '😛': '', '😜': '', '🤪': '',
    '😝': '', '🤗': '', '🤭': '', '🤫': '', '🤔': '',
    '😐': '', '😑': '', '😶': '', '😏': '', '😒': '',
    '🙄': '', '😬': '', '😮': '', '😯': '', '😲': '',
    '😳': '', '🥺': '', '😢': '', '😭': '', '😤': '',
    '😠': '', '😡': '', '🤬': '', '😈': '', '👿': '',
    '👍': '', '👎': '', '👏': '', '🙏': '',
    '❤️': '', '💯': '', '⭐': '', '🌟': '',
    '✅': '', '❌': '', '✨': '', '🔥': '',
    '🥴': '', '💪': '', '😎': '',
}


def normalize_text(text):
    """Normalisasi teks review untuk ABSA."""
    if pd.isna(text) or not isinstance(text, str):
        return ''

    # 1. Hapus emoji
    for emoji, replacement in EMOJI_SENTIMENT.items():
        text = text.replace(emoji, replacement)
    # Hapus sisa emoji yang tidak ada di mapping
    text = emoji_pattern.sub('', text)

    # 2. Hapus karakter non-printable & kontrol Unicode
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f-\x9f]', '', text)

    # 3. Normalisasi tanda kutip & apostrof
    text = text.replace('\u201c', '"').replace('\u201d', '"')  # smart quotes
    text = text.replace('\u2018', "'").replace('\u2019', "'")  # smart apostrophe
    text = text.replace('\u2014', ' - ')  # em dash
    text = text.replace('\u2013', ' - ')  # en dash

    # 4. Normalisasi whitespace
    text = re.sub(r'\n+', ' ', text)  # newline -> spasi
    text = re.sub(r'\t+', ' ', text)  # tab -> spasi
    text = re.sub(r' {2,}', ' ', text)  # multiple spaces -> single

    # 5. Hapus spasi di awal/akhir
    text = text.strip()

    return text


# Apply
print('Menerapkan normalisasi teks...')
df['text_review'] = df['text_review'].apply(normalize_text)

# Hapus review yang jadi kosong setelah normalisasi
before = len(df)
df = df[df['text_review'].str.strip().astype(bool)]
df = df[df['text_review'].str.len() >= MIN_LENGTH]
removed = before - len(df)
print(f'Review kosong setelah normalisasi: {removed:,} dihapus')
print(f'Sisa: {len(df):,} review')

# Sample
print('\nSample setelah normalisasi:')
for _, row in df.head(5).iterrows():
    preview = row['text_review'][:100] + '...' if len(row['text_review']) > 100 else row['text_review']
    print(f'  [{row["platform"]}] {preview}')


### Case Folding


In [ ]:
df['text_review'] = df['text_review'].str.lower()
print('Case folding selesai.')

# Sample
for _, row in df.head(3).iterrows():
    preview = row['text_review'][:100] + '...' if len(row['text_review']) > 100 else row['text_review']
    print(f'  {preview}')


### Normalisasi Slang dan Singkatan


In [ ]:
# Kamus normalisasi slang/singkatan Indonesia
SLANG_DICT = {
    # Singkatan umum
    'yg': 'yang',
    'dgn': 'dengan',
    'dg': 'dengan',
    'utk': 'untuk',
    'krn': 'karena',
    'tp': 'tapi',
    'tdk': 'tidak',
    'gak': 'tidak',
    'ga': 'tidak',
    'gk': 'tidak',
    'nggak': 'tidak',
    'enggak': 'tidak',
    'trs': 'terus',
    'trus': 'terus',
    'bgt': 'banget',
    'bngt': 'banget',
    'bkn': 'bukan',
    'blm': 'belum',
    'sdh': 'sudah',
    'udh': 'sudah',
    'udah': 'sudah',
    'lg': 'lagi',
    'lgi': 'lagi',
    'jg': 'juga',
    'jgn': 'jangan',
    'sm': 'sama',
    'dr': 'dari',
    'dlm': 'dalam',
    'dpt': 'dapat',
    'hrs': 'harus',
    'spy': 'supaya',
    'ttg': 'tentang',
    'org': 'orang',
    'sy': 'saya',
    'ak': 'aku',
    'kmu': 'kamu',
    'km': 'kamu',
    'bs': 'bisa',
    'cm': 'cuma',
    'kl': 'kalau',
    'klo': 'kalau',
    'kalo': 'kalau',
    'mkn': 'mungkin',
    'mgkn': 'mungkin',
    'emg': 'memang',
    'emang': 'memang',
    'bbrp': 'beberapa',
    'brg': 'barang',
    'bgmn': 'bagaimana',
    'gmn': 'gimana',
    'dmn': 'dimana',
    'kmr': 'kamar',
    'tmpt': 'tempat',
    'smua': 'semua',
    'bnr': 'benar',
    'bner': 'benar',
    'ckp': 'cukup',
    'scr': 'secara',
    'trm': 'terima',
    'trmksh': 'terima kasih',
    'thx': 'terima kasih',
    'thanks': 'terima kasih',
    'thank': 'terima kasih',
    'tq': 'terima kasih',
    'makasih': 'terima kasih',
    'mksh': 'terima kasih',
    'mks': 'terima kasih',
    # Slang ekspresif
    'mantap': 'bagus',
    'mantapp': 'bagus',
    'mantapppp': 'bagus',
    'mantul': 'bagus',
    'keren': 'bagus',
    'oke': 'baik',
    'okee': 'baik',
    'okeee': 'baik',
    'okeeee': 'baik',
    'okey': 'baik',
    'ok': 'baik',
    'okay': 'baik',
    'okelah': 'baik',
    'okeh': 'baik',
    'jelek': 'buruk',
    'jlk': 'buruk',
    'ancur': 'buruk',
    'parah': 'buruk',
    'zonk': 'buruk',
    # Code-switching umum (kata Inggris di review hotel ID)
    'bfast': 'breakfast',
    'b-fast': 'breakfast',
    'brekfas': 'breakfast',
    'breskfast': 'breakfast',
    'breakfas': 'breakfast',
    'good': 'bagus',
    'great': 'bagus',
    'nice': 'bagus',
    'bad': 'buruk',
    'helpful': 'membantu',
    'friendly': 'ramah',
    'comfortable': 'nyaman',
    'clean': 'bersih',
    'staff': 'staf',
    # Domain hotel
    'ac': 'air conditioner',
    'wifi': 'wi-fi',
    'wf': 'wi-fi',
    'tv': 'televisi',
}


def normalize_slang(text):
    """Ganti singkatan/slang dengan kata baku."""
    words = text.split()
    result = []
    for word in words:
        # Bersihkan tanda baca di akhir kata untuk matching
        clean_word = re.sub(r'[.,!?;:]+$', '', word)
        trailing = word[len(clean_word):]

        if clean_word in SLANG_DICT:
            result.append(SLANG_DICT[clean_word] + trailing)
        else:
            result.append(word)
    return ' '.join(result)


print('Menerapkan normalisasi slang...')
df['text_review'] = df['text_review'].apply(normalize_slang)
print('Selesai.')

# Sample
print('\nSample setelah normalisasi slang:')
for _, row in df.sample(5, random_state=42).iterrows():
    preview = row['text_review'][:120] + '...' if len(row['text_review']) > 120 else row['text_review']
    print(f'  {preview}')


### Normalisasi Karakter Berulang


In [ ]:
def normalize_repeated_chars(text):
    """Kurangi karakter berulang > 2 menjadi 1.
    Contoh: enakkkkk -> enak, baguuusss -> bagus"""
    return re.sub(r'(.)\1{2,}', r'\1', text)


# Cek sample sebelum
repeated = df[df['text_review'].str.contains(r'(.)\1{2,}', regex=True)]
print(f'Review dengan karakter berulang: {len(repeated):,}')
print('Sample sebelum:')
for _, row in repeated.head(5).iterrows():
    # Find the repeated part
    matches = re.findall(r'\S*(.)\1{2,}\S*', row['text_review'])
    words_with_repeat = re.findall(r'\S*(.)\1{2,}\S*', row['text_review'])
    preview = row['text_review'][:80]
    print(f'  {preview}')

# Apply
df['text_review'] = df['text_review'].apply(normalize_repeated_chars)
print(f'\nNormalisasi karakter berulang selesai.')

print('\nSample setelah:')
for _, row in df.loc[repeated.head(5).index].iterrows():
    preview = row['text_review'][:80]
    print(f'  {preview}')


### Membersihkan Tanda Baca


In [ ]:
def clean_punctuation(text):
    """Bersihkan tanda baca berlebihan tapi pertahankan yang bermakna."""
    # Hapus multiple punctuation: "!!!" -> "!", "..." -> "."
    text = re.sub(r'([!?.]){2,}', r'\1', text)

    # Hapus tanda baca dekoratif
    text = re.sub(r'[~*#@^&|\\{}\[\]<>]+', ' ', text)

    # Normalisasi spasi setelah tanda baca
    text = re.sub(r'\s*([.,!?;:])\s*', r'\1 ', text)

    # Normalisasi spasi
    text = re.sub(r' {2,}', ' ', text)
    text = text.strip()

    return text


df['text_review'] = df['text_review'].apply(clean_punctuation)
print('Normalisasi tanda baca selesai.')


### Final Cleaning dan Near-Duplicate Check


In [ ]:
# Cek near-duplicates setelah normalisasi
before = len(df)
df = df.drop_duplicates(subset='text_review', keep='first')
removed = before - len(df)
print(f'Near-duplicates dihapus: {removed:,}')

# Final filter panjang minimal
before = len(df)
df = df[df['text_review'].str.strip().str.len() >= MIN_LENGTH]
removed = before - len(df)
print(f'Review terlalu pendek setelah cleaning: {removed:,} dihapus')

print(f'\nDataset bersih: {len(df):,} review')


### Export Clean Dataset


In [ ]:
# Export dataset clean. Kolom disamakan dengan dataset yang dipakai untuk labeling/training.
df = df.reset_index(drop=True)
df['review_id'] = range(1, len(df) + 1)
output_cols = ['review_id', 'platform', 'hotel_name', 'text_review', 'text_review_original', 'date']
df_out = df[output_cols]
clean_output_path = os.path.join(BASE_DIR, OUTPUT_FILE_CLEAN)
df_out.to_csv(clean_output_path, index=False, encoding='utf-8-sig')
print(f'Clean dataset saved: {clean_output_path}')
print(f'Clean rows: {len(df_out):,}')
print(f'Removed rows: {initial_count - len(df_out):,}')
if len(df_out) != EXPECTED_CLEAN_ROWS:
    raise ValueError(f'Clean row count mismatch: expected {EXPECTED_CLEAN_ROWS:,}, actual {len(df_out):,}')
print('Jumlah baris sudah sesuai baseline labeling/training.')
display(df_out.head())


## EDA Processed + WordCloud

Tahap ini dilakukan setelah dataset clean terbentuk. WordCloud dipindahkan ke tahap ini agar visualisasi dibuat dari data yang sudah diproses, bukan dari data raw.


### Load Clean Dataset untuk EDA Processed


In [ ]:
from collections import Counter
from wordcloud import WordCloud

processed_df = df_out.copy()
EXPECTED_CLEAN_ROWS = 14747
print('=== EDA Processed Dataset ===')
print(f'Total review clean: {len(processed_df):,}')
print(f'Kolom: {processed_df.columns.tolist()}')
if len(processed_df) != EXPECTED_CLEAN_ROWS:
    print(f'[WARN] Expected {EXPECTED_CLEAN_ROWS:,}, actual {len(processed_df):,}')

display(processed_df.head())


### Overview Clean Dataset


In [ ]:
processed_dates = pd.to_datetime(processed_df['date'], errors='coerce')
print(f'Rentang tanggal: {processed_dates.min().date()} s/d {processed_dates.max().date()}')
print(f'Tanggal invalid: {processed_dates.isna().sum():,}')

processed_platform_summary = processed_df.groupby('platform').size().to_frame('jumlah_review').reset_index().sort_values('jumlah_review', ascending=False)
processed_hotel_summary = processed_df.groupby('hotel_name').size().to_frame('jumlah_review').reset_index().sort_values('jumlah_review', ascending=False)
processed_platform_hotel = pd.crosstab(processed_df['hotel_name'], processed_df['platform'], margins=True)

display(processed_platform_summary)
display(processed_hotel_summary)
display(processed_platform_hotel)


### Kualitas Teks Clean


In [ ]:
processed_text_len = processed_df['text_review'].fillna('').astype(str).str.len()
processed_text_summary = pd.DataFrame({
    'metric': ['min_len', 'mean_len', 'median_len', 'max_len', 'duplicate_text', 'short_review_lt_20'],
    'value': [
        processed_text_len.min(),
        round(processed_text_len.mean(), 2),
        processed_text_len.median(),
        processed_text_len.max(),
        processed_df['text_review'].duplicated().sum(),
        (processed_text_len < 20).sum(),
    ],
})
display(processed_text_summary)

processed_years = pd.to_datetime(processed_df['date'], errors='coerce').dt.year.value_counts().sort_index()
ax = processed_years.plot(kind='bar', figsize=(12, 5), color='#4472C4')
ax.set_title('Distribusi Review per Tahun - Processed')
ax.set_xlabel('Tahun')
ax.set_ylabel('Jumlah Review')
plt.tight_layout()
plt.show()


### Tokenisasi dan Frekuensi Kata


In [ ]:
STOPWORDS_ID = {
    'yang','dan','di','ke','dari','ini','itu','untuk','dengan','pada','adalah','sebagai','dalam','karena',
    'saya','kami','kita','mereka','nya','ada','tidak','bukan','atau','juga','sudah','belum','akan','lebih',
    'sangat','cukup','sekali','banget','lagi','bisa','dapat','jadi','kalau','kalo','saat','selama','setelah',
    'sebelum','hotel','santika','review','ulasan','kamar','tempat','orang','hari','malam','menggunakan',
    'the','and','for','with','this','that','was','were','are','is','to','of','in','on','at','it','we','i',
}

def tokenize_processed(text):
    words = re.findall(r'[a-zA-Z\u00C0-\u024F]+', str(text).lower())
    return [word for word in words if word not in STOPWORDS_ID and len(word) > 2]

processed_words = []
for text in processed_df['text_review'].dropna().astype(str):
    processed_words.extend(tokenize_processed(text))

processed_word_counts = Counter(processed_words).most_common(30)
processed_word_freq_df = pd.DataFrame(processed_word_counts, columns=['word', 'frequency'])
display(processed_word_freq_df)


### WordCloud Semua Review Processed


In [ ]:
fig_dir = Path(BASE_DIR) / 'eda_processed_figures'
fig_dir.mkdir(parents=True, exist_ok=True)

all_processed_text = ' '.join(processed_df['text_review'].dropna().astype(str).tolist())
wc = WordCloud(
    width=1400,
    height=700,
    background_color='white',
    stopwords=STOPWORDS_ID,
    colormap='viridis',
    max_words=180,
    min_font_size=8,
    max_font_size=90,
    collocations=False,
    random_state=42,
).generate(all_processed_text)

fig, ax = plt.subplots(figsize=(16, 8))
ax.imshow(wc, interpolation='bilinear')
ax.set_title('WordCloud - Semua Review Hotel Santika (Processed)', fontsize=18, fontweight='bold')
ax.axis('off')
plt.tight_layout()
fig_path = fig_dir / 'wordcloud_all_processed.png'
plt.savefig(fig_path, dpi=200, bbox_inches='tight')
plt.show()
print('Saved:', fig_path)


### WordCloud per Platform


In [ ]:
platforms = sorted(processed_df['platform'].dropna().unique())
colormaps = {'Agoda': 'Blues', 'Tiket': 'Oranges', 'Traveloka': 'Greens'}

fig, axes = plt.subplots(1, len(platforms), figsize=(6 * len(platforms), 5))
if len(platforms) == 1:
    axes = [axes]

for ax, platform in zip(axes, platforms):
    text = ' '.join(processed_df[processed_df['platform'] == platform]['text_review'].dropna().astype(str).tolist())
    wc = WordCloud(
        width=800,
        height=500,
        background_color='white',
        stopwords=STOPWORDS_ID,
        colormap=colormaps.get(platform, 'viridis'),
        max_words=100,
        collocations=False,
        random_state=42,
    ).generate(text)
    count = len(processed_df[processed_df['platform'] == platform])
    ax.imshow(wc, interpolation='bilinear')
    ax.set_title(f'{platform}\n({count:,} review)', fontsize=12, fontweight='bold')
    ax.axis('off')

plt.suptitle('WordCloud per Platform - Processed', fontsize=16, fontweight='bold')
plt.tight_layout()
fig_path = fig_dir / 'wordcloud_by_platform_processed.png'
plt.savefig(fig_path, dpi=200, bbox_inches='tight')
plt.show()
print('Saved:', fig_path)


### WordCloud per Hotel


In [ ]:
hotels = sorted(processed_df['hotel_name'].dropna().unique())
cols = 3
rows = (len(hotels) + cols - 1) // cols
hotel_cmaps = ['viridis', 'plasma', 'cividis', 'magma', 'inferno', 'twilight']

fig, axes = plt.subplots(rows, cols, figsize=(18, 5 * rows))
axes_flat = axes.flatten() if hasattr(axes, 'flatten') else [axes]

for i, hotel in enumerate(hotels):
    text = ' '.join(processed_df[processed_df['hotel_name'] == hotel]['text_review'].dropna().astype(str).tolist())
    wc = WordCloud(
        width=800,
        height=500,
        background_color='white',
        stopwords=STOPWORDS_ID,
        colormap=hotel_cmaps[i % len(hotel_cmaps)],
        max_words=100,
        collocations=False,
        random_state=42,
    ).generate(text)
    count = len(processed_df[processed_df['hotel_name'] == hotel])
    short_name = hotel.replace('Hotel Santika ', '')
    axes_flat[i].imshow(wc, interpolation='bilinear')
    axes_flat[i].set_title(f'{short_name}\n({count:,} review)', fontsize=11, fontweight='bold')
    axes_flat[i].axis('off')

for j in range(len(hotels), len(axes_flat)):
    axes_flat[j].axis('off')

plt.suptitle('WordCloud per Hotel - Processed', fontsize=16, fontweight='bold')
plt.tight_layout()
fig_path = fig_dir / 'wordcloud_by_hotel_processed.png'
plt.savefig(fig_path, dpi=200, bbox_inches='tight')
plt.show()
print('Saved:', fig_path)


### Top Words dan Export Ringkasan Processed


In [ ]:
if not processed_word_freq_df.empty:
    top_words = processed_word_freq_df.head(20).iloc[::-1]
    fig, ax = plt.subplots(figsize=(11, 7))
    ax.barh(top_words['word'], top_words['frequency'], color='#4472C4')
    ax.set_title('Top 20 Kata Paling Sering Muncul - Processed', fontsize=15, fontweight='bold')
    ax.set_xlabel('Frekuensi')
    plt.tight_layout()
    fig_path = fig_dir / 'top_words_processed.png'
    plt.savefig(fig_path, dpi=200, bbox_inches='tight')
    plt.show()
    print('Saved:', fig_path)

processed_summary_path = Path(BASE_DIR) / 'eda_processed_summary.xlsx'
with pd.ExcelWriter(processed_summary_path) as writer:
    processed_platform_summary.to_excel(writer, index=False, sheet_name='platform')
    processed_hotel_summary.to_excel(writer, index=False, sheet_name='hotel')
    processed_platform_hotel.to_excel(writer, sheet_name='platform_x_hotel')
    processed_text_summary.to_excel(writer, index=False, sheet_name='text_quality')
    processed_word_freq_df.to_excel(writer, index=False, sheet_name='top_words')
print('Saved:', processed_summary_path)
